In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib

from functools import partial
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
# rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

import jax
import jax.numpy as jnp
import jax_dataclasses as jdc
from jax.tree_util import tree_flatten, tree_unflatten

# jax.config.update('jax_platform_name', 'cpu')

# jax.config.update("jax_debug_nans", True)

gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax

In [ ]:
from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import extract_metrics_over_timesteps, extract_metrics_over_timesteps_via_interpolation
from dmpe.evaluation.plotting_utils import plot_metrics_by_sequence_length_for_all_algos
from dmpe.evaluation.experiment_utils import get_experiment_ids

from dmpe.utils.density_estimation import select_bandwidth
from dmpe.evaluation.experiment_utils import default_jsd, default_ae, default_mcudsa, default_ksfc, default_df

In [ ]:
from math import e

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

import matplotlib.ticker as ticker

def custom_formatter(val, pos):
    if val < 1.0:
        return rf"${val:.2f}$"  
    else:
        return rf"${val:.1f}$" 

def plot_metrics_by_sequence_length_for_all_algos(data_per_algo, lengths, algo_names, use_log=False, plot_log=False):
    assert len(data_per_algo) == len(algo_names), "Mismatch in number of algo results and number of algo names"

    metric_keys = data_per_algo[0].keys()

    fig, axs = plt.subplots(len(metric_keys), figsize=(half_column_width, 0.7 * 11 / 4 * len(metric_keys)), sharex=True) # figsize=(19, 18)
    colors = plt.rcParams["axes.prop_cycle"]()

    for algo_name, data in zip(algo_names, data_per_algo):
        c = next(colors)["color"]
        if c == '#d62728':
            c = next(colors)["color"]

        for metric_idx, metric_key in enumerate(metric_keys):

            mean = jnp.nanmean(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanmean(data[metric_key], axis=0)
            std = jnp.nanstd(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanstd(data[metric_key], axis=0)

            if algo_name=="$\mathrm{DMPE}$":
                style = "dashed"
            elif algo_name=="$\mathrm{iGOATS}$":
                style = "dashdot"
            else:
                style=None
            
            axs[metric_idx].plot(
                lengths,
                mean, 
                label=algo_name if metric_idx == 0 else None,
                color=c,
                #linestyle='dashed' if algo_name=="$\mathrm{DMPE}$" else None,
                linewidth=2.5,
                linestyle=style,
            )
            axs[metric_idx].fill_between(
                lengths,
                mean - std,
                mean + std,
                color=c,
                alpha=0.1,
            )

    if plot_log:
        for ax in axs:
            ax.set_yscale('log', base=10)

        for idx_y, ax in enumerate(axs[:-1]):
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
            ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
            ax.yaxis.set_major_locator(ticker.LogLocator(numticks=2))

            if idx_y > 0:
                ax.yaxis.set_minor_locator(ticker.LogLocator(subs=(0.5,), numticks=4))
            else:
                ax.yaxis.set_minor_locator(ticker.LogLocator(subs=(0.3, 0.6), numticks=4))
            #ax.yaxis.set_minor_locator(ticker.LogLocator(subs="auto"))
    
    for idx, metric_key in enumerate(metric_keys):
        axs[idx].set_ylabel(f"$\mathcal{{L}}_\mathrm{{{metric_key.upper()}}}$")

    axs[-1].set_xlabel("$k$")
    axs[-1].set_xlim(lengths[0], lengths[-1])

    [ax.grid(True, which="both", alpha=0.3) for ax in axs]
    
    legend = fig.legend(
        prop={'size': 5 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, -0.02),
        fancybox=True,
        shadow=False,
        ncol=len(algo_names)
    )

    plt.subplots_adjust(hspace=0.02)
    
    plt.tight_layout(pad=0.05)

    [ax.tick_params(axis="y", direction='in') for ax in axs]
    [ax.tick_params(axis="x", direction='in') for ax in axs]
    # [ax.yaxis.set_major_locator(plt.MaxNLocator(3)) for ax in axs]

    fig.align_ylabels(axs)

    return fig

In [ ]:
lengths = jnp.linspace(1000, 15000, 15, dtype=jnp.int32)
lengths

In [ ]:
def extract_results(lengths, raw_results_path, algo_names, interpolate_to_lengths, system_name, metrics=None, extra_folders=None):

    all_results_by_metric = {}
    
    for (algo_name, use_interpolation) in zip(algo_names, interpolate_to_lengths):
        full_results_path = raw_results_path / pathlib.Path(algo_name) / pathlib.Path(system_name)
        full_results_path = full_results_path / pathlib.Path(extra_folders) if extra_folders is not None else full_results_path

        print("Extract results for", algo_name, "\n at", full_results_path)

        if not use_interpolation:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                lengths=lengths,
                metrics=metrics,
            )
        else:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps_via_interpolation(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                target_lengths=lengths,
                metrics=metrics,
            )
        print("\n")
    return all_results_by_metric

In [ ]:
from dmpe.utils.density_estimation import build_grid

In [ ]:
points_per_dim = 7
dim = 5

support_points = build_grid(dim, low=-1, high=1, points_per_dim=points_per_dim)
support_spacing = jnp.abs(support_points[0] - support_points[1])[-1] / 2
print(support_spacing)

## fluid_tank:

In [ ]:
# system_name = "fluid_tank"

# all_fluid_tank_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics={
#         "jsd": partial(default_jsd, points_per_dim=50, bandwidth=select_bandwidth(2, 2, 50, 0.3).item()),
#         "ae": default_ae,
#         "mcudsa": partial(default_mcudsa, points_per_dim=50),
#         "ksfc": partial(default_ksfc, points_per_dim=50, eps=1e-6),
#         "df": partial(default_df, points_per_dim=50),
#     }
# )
# with open(DataPaths().se_cs_experiments / "fluid_tank_results.pickle", "wb") as handle:
#     pickle.dump(all_fluid_tank_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
system_name = "fluid_tank"

with open(DataPaths().se_cs_experiments / "fluid_tank_results.pickle", 'rb') as handle:
    all_fluid_tank_results_by_metric = pickle.load(handle)

print(all_fluid_tank_results_by_metric.keys())

pm_dmpe_results_by_metric = all_fluid_tank_results_by_metric["perfect_model_dmpe"]
dmpe_results_by_metric = all_fluid_tank_results_by_metric["dmpe"]
sgoats_results_by_metric = all_fluid_tank_results_by_metric["sgoats"]#["interp"]
igoats_results_by_metric = all_fluid_tank_results_by_metric["igoats"]#["interp"]
random_walk_results_by_metric = all_fluid_tank_results_by_metric["random_walk"]

plot_metrics_by_sequence_length_for_all_algos(
    data_per_algo=[pm_dmpe_results_by_metric, dmpe_results_by_metric, sgoats_results_by_metric, igoats_results_by_metric, random_walk_results_by_metric],
    lengths=lengths,
    algo_names=["$\mathrm{PM-DMPE}$", "$\mathrm{DMPE}$", "$\mathrm{sGOATS}$", "$\mathrm{iGOATS}$", "$\mathrm{random-walk}$"],
    use_log=False,
    plot_log=True,
);
plt.savefig(f"metrics_per_sequence_length_{system_name}.pdf", bbox_inches='tight')

## pendulum:

In [ ]:
# system_name = "pendulum"

# all_pendulum_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics={
#         "jsd": partial(default_jsd, points_per_dim=50, bandwidth=select_bandwidth(2, 3, 50, 0.3)),
#         "ae": default_ae,
#         "mcudsa": partial(default_mcudsa, points_per_dim=50),
#         "ksfc": partial(default_ksfc, points_per_dim=50, eps=1e-6),
#         "df": partial(default_df, points_per_dim=15),
#     }
# )
# with open(DataPaths().se_cs_experiments / "pendulum_results.pickle", "wb") as handle:
#     pickle.dump(all_pendulum_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
system_name = "pendulum"

with open(DataPaths().se_cs_experiments / "pendulum_results.pickle", 'rb') as handle:
    all_pendulum_results_by_metric = pickle.load(handle)

print(all_pendulum_results_by_metric.keys())

pm_dmpe_results_by_metric = all_pendulum_results_by_metric["perfect_model_dmpe"]
dmpe_results_by_metric = all_pendulum_results_by_metric["dmpe"]
sgoats_results_by_metric = all_pendulum_results_by_metric["sgoats"]#["interp"]
igoats_results_by_metric = all_pendulum_results_by_metric["igoats"]#["interp"]
random_walk_results_by_metric = all_pendulum_results_by_metric["random_walk"]#["interp"]

plot_metrics_by_sequence_length_for_all_algos(
    data_per_algo=[
        pm_dmpe_results_by_metric,
        dmpe_results_by_metric,
        sgoats_results_by_metric,
        igoats_results_by_metric,
        random_walk_results_by_metric,
    ],
    lengths=lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
    use_log=False,
    plot_log=True,
);
plt.savefig(f"metrics_per_sequence_length_{system_name}.pdf", bbox_inches='tight')

## cart pole:

In [ ]:
# system_name = "cart_pole"

# all_cart_pole_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics={
#         "jsd": partial(default_jsd, points_per_dim=20, bandwidth=select_bandwidth(2, 5, 20, 0.1)),
#         "ae": default_ae,
#         "mcudsa": partial(default_mcudsa, points_per_dim=20),
#         "ksfc": partial(default_ksfc, points_per_dim=20, variance=0.1, eps=1e-6),
#         "df": partial(default_df, points_per_dim=7),
#     }
# )
# with open(DataPaths().se_cs_experiments / "cart_pole_results.pickle", "wb") as handle:
#     pickle.dump(all_cart_pole_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
system_name = "cart_pole"
with open(DataPaths().se_cs_experiments / "cart_pole_results.pickle", 'rb') as handle:
    all_cart_pole_results_by_metric = pickle.load(handle)

print(all_cart_pole_results_by_metric.keys())

pm_dmpe_results_by_metric = all_cart_pole_results_by_metric["perfect_model_dmpe"]
dmpe_results_by_metric = all_cart_pole_results_by_metric["dmpe"]
sgoats_results_by_metric = all_cart_pole_results_by_metric["sgoats"]#["interp"]
igoats_results_by_metric = all_cart_pole_results_by_metric["igoats"]#["interp"]
random_walk_results_by_metric = all_cart_pole_results_by_metric["random_walk"]

plot_metrics_by_sequence_length_for_all_algos(
    data_per_algo=[
        pm_dmpe_results_by_metric,
        dmpe_results_by_metric,
        sgoats_results_by_metric,
        igoats_results_by_metric,
        random_walk_results_by_metric
    ],
    lengths=lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
    use_log=False,
);
plt.savefig(f"metrics_per_sequence_length_{system_name}.pdf", bbox_inches='tight')

# Plot all together:

In [ ]:
all_fluid_tank_results_by_metric.keys()
all_pendulum_results_by_metric.keys()
all_cart_pole_results_by_metric.keys()

In [ ]:
all_results = dict(
    fluid_tank=all_fluid_tank_results_by_metric,
    pendulum=all_pendulum_results_by_metric,
    cart_pole=all_cart_pole_results_by_metric,
)

In [ ]:
lengths

In [ ]:
def plot_metrics_by_sequence_length_for_all_algos_for_all_systems(
    data_per_algo_per_system,
    lengths,
    algo_names,
    plot_log=True,
):
    systems = list(data_per_algo_per_system.keys())
    metric_keys = list(list(data_per_algo_per_system[systems[0]].values())[0].keys())


    print(systems, metric_keys)
    fig, axs = plt.subplots(len(metric_keys), len(systems), figsize=(full_column_width, 13), sharex=True)

    for sys_idx, system_name in enumerate(systems):
        data_per_algo = data_per_algo_per_system[system_name]
        colors = plt.rcParams["axes.prop_cycle"]()

        for algo_name, data in zip(algo_names, data_per_algo.values()):
            c = next(colors)["color"]
            if c == '#d62728':
                c = next(colors)["color"]
            for metric_idx, metric_key in enumerate(metric_keys):
                
                mean = jnp.nanmean(data[metric_key], axis=0)
                std = jnp.nanstd(data[metric_key], axis=0)


                if algo_name=="$\mathrm{sGOATS}$":
                    style = "dotted"
                elif algo_name=="$\mathrm{iGOATS}$":
                    style = "dashdot"
                elif algo_name=="$\mathrm{DMPE}$":
                    style = "dashed"
                else:
                    style=None
                
                axs[metric_idx, sys_idx].plot(
                    lengths,
                    mean,  # jnp.log(mean) if use_log else mean,
                    label=algo_name if metric_idx == 0 and sys_idx == 0 else None,
                    color=c,
                    linewidth=2.5,
                    linestyle=style,
                )
                axs[metric_idx, sys_idx].fill_between(
                    lengths,
                    mean - std,  # jnp.log(mean - std) if use_log else mean - std,
                    mean + std,  # jnp.log(mean + std) if use_log else mean + std,
                    color=c,
                    alpha=0.1,
                )

    for idx, metric_key in enumerate(metric_keys):
        axs[idx, 0].set_ylabel(f"$\mathcal{{L}}_\mathrm{{{metric_key.upper()}}}$")

    for ax in axs[-1]:
        ax.set_xlabel("$k$")

    for ax_ in axs:
        for ax in ax_:
            ax.grid(True, which="both", alpha=0.3)
            ax.tick_params(which='both', axis="y", direction='in')
            ax.tick_params(which='both', axis="x", direction='in') 
        ax_[-1].set_xlim(lengths[0] - 0.02 * lengths[-1], lengths[-1] + 0.02 * lengths[-1])

    legend = fig.legend(
        prop={'size': 8 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, 0.0),
        fancybox=True,
        shadow=False, 
        ncol=len(algo_names)
    )

    for ax, col in zip(axs[0], ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]):
        ax.set_title(col)

    if plot_log:
        for ax_ in axs[:-1]:
            for ax in ax_:
                ax.set_yscale('log', base=10)

        for idx_y, ax_ in enumerate(axs[:-1]):
            for idx_x, ax in enumerate(ax_):


                if idx_y == 0:
                    # jsd
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.05, 0.1, 0.3])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_major_locator(ticker.LogLocator(numticks=2))
                        ax.yaxis.set_minor_locator(ticker.LogLocator(subs=(0.3, 0.6), numticks=2))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                    pass
                elif idx_y == 1:
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([7, 25, 50])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([2, 5, 10])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([1, 2, 3])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                elif idx_y == 2:
                    # MCUDSA
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.02, 0.05, 0.1])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.1, 0.5])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.4, 0.5, 0.7])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                elif idx_y == 3:
                    # KSFC
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([10, 15, 25])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_ticks([10, 100])
                        ax.set_ylim(5, 200)
                        ax.yaxis.set_minor_locator(ticker.LogLocator(numticks=3))
                    elif idx_x == 2:
                        ax.yaxis.set_ticks([1000, 10_000])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                else:
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.25, 0.5, 0.75])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.25, 0.5, 0.75])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.8, 0.9, 1.0])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))

    fig.tight_layout(h_pad=-0.1, w_pad=0.35)
    fig.align_ylabels(axs)
    return fig, legend

In [ ]:
all_cart_pole_results_by_metric.keys()

In [ ]:
targeted_algo_order = ["perfect_model_dmpe", "dmpe", "sgoats", "igoats", "random_walk"]
for sys_name, sys_results in all_results.items():
    all_results[sys_name] = {algo_key: sys_results[algo_key] for algo_key in targeted_algo_order}

In [ ]:
plot_metrics_by_sequence_length_for_all_algos_for_all_systems(
    all_results,
    lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
)

#plt.show()
plt.savefig("all_metrics_all_systems_all_algos.pdf", bbox_inches='tight')